In [18]:
# import rpy2.robjects as ro

# # Source the R script
# ro.r['source']('downsample.R')

# # Get R function into Python
# downsample_clonotypes = ro.globalenv['downsample_clonotypes']

# # Call the function with your parameters
# downsample_clonotypes(num_clonotypes=10000, threshold=10000, prob=True)

In [19]:
import chephy_model as cpm
import gnn_utils
import torch
import pandas as pd


In [20]:
atchley_path = "/home/dsi/chenbis/repos/sol_lab/files/atchley.csv"
atchley_df = pd.read_csv(atchley_path)
atchley_dict = atchley_df.set_index("amino.acid").to_dict(orient="index")
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

downsample_size = 5000

downsampled_path = f"/dsi/efroni-lab/sbm/chen2/downsampled_data/ovarian/{downsample_size}/"

graph_path = f"/dsi/efroni-lab/sbm/chen2/graph_data/ovarian/{downsample_size}"

In [21]:
from pathlib import Path
file_names = [downsampled_path + f.name for f in Path(downsampled_path).iterdir() if f.is_file()]

In [22]:
def prepare_data(data, cdr3_header):

    # remove cdr3 sequences that contain non-aa letters (this also removes nan)
    amino_acid_pattern = r'^[ARNDCEQGHILKMFPSTWYV]+$'

    data_filtered = data[data[cdr3_header].str.match(amino_acid_pattern, case=False, na=False)]

    data_filtered = data_filtered[data_filtered[cdr3_header].str.len() >= 8]

    return data_filtered

In [23]:
from scipy.spatial.distance import squareform 


def get_graph_data(data, cdr3_header, label, distance_threshold = 0.2, use_mst=True, enrich=False):


    data = prepare_data(data, cdr3_header)    
    data = cpm.truncate_sequences(data, cdr3_header, 4, 4)

    sequences_set = set(data["cdr3_truncated"].dropna())

    chephy_matrix, sequences_list = cpm.find_che_phy_dist(sequences_set)

    

    idx_map = {s: i for i, s in enumerate(sequences_list)}
    data["tcr_index"] = data["cdr3_truncated"].map(idx_map)

    
    # pick ONE representative full sequence per node index (first occurrence)
    node_df = (
        data.dropna(subset=["tcr_index"])
            .sort_values("tcr_index")
            .drop_duplicates(subset=["tcr_index"])
    )
    node_sequences = node_df.set_index("tcr_index")[cdr3_header].reindex(range(len(sequences_list))).tolist()

    # sum weights per node index, aligned to [0..num_nodes-1]
    node_weights = (
        data.dropna(subset=["tcr_index"])
            .groupby("tcr_index")["readFraction"].sum()
            .reindex(range(len(sequences_list)))
            .fillna(0.0)
            .to_numpy()
            .tolist()
    )

    graph = gnn_utils.build_graph(
        squareform(chephy_matrix),
        node_sequences,              # <-- now length == num nodes
        label=label,
        node_weights=node_weights,   # <-- now same length
        distance_threshold=distance_threshold,
        use_mst=use_mst,
        enrich=enrich
    )

    return graph, data


In [24]:
from tqdm import tqdm
import os

cdr3_header = "aaSeqCDR3"

distance_threshold=0.1
use_mst=True
enrich=True

sub_folder = f"{graph_path}/{distance_threshold}_{use_mst}_{enrich}"

if not os.path.exists(sub_folder):
    os.makedirs(sub_folder)


for file_name in tqdm(file_names, desc="Processing files"):
    df = pd.read_csv(file_name, sep="\t")
    label = file_name.split("/")[-1].split("_")[0]
    file_save_name = f"{sub_folder}/{file_name.rsplit('/', 1)[-1]}.pt"

    if label == "sick":
        label = 1
    else:
        label = 0
        
    data = get_graph_data(df, cdr3_header, label=label, distance_threshold=distance_threshold, use_mst=use_mst, enrich=enrich)

    torch.save(data, file_save_name)


Processing files: 100%|██████████| 82/82 [09:30<00:00,  6.95s/it]


In [25]:
# Calculate average chephy distance for each patient
patient_distances = []

for file_name in file_names:
    df = pd.read_csv(file_name, sep="\t")
    label = file_name.split("/")[-1].split("_")[0]
    patient_id = file_name.split("/")[-1].split("_")[1]
    
    # Prepare data
    data_filtered = prepare_data(df, cdr3_header)
    data_filtered = cpm.truncate_sequences(data_filtered, cdr3_header, 4, 4)
    
    sequences_set = set(data_filtered["cdr3_truncated"].dropna())
    
    if len(sequences_set) > 1:
        chephy_matrix, sequences_list = cpm.find_che_phy_dist(sequences_set)
        avg_distance = chephy_matrix.mean()
    else:
        avg_distance = 0.0
    
    patient_distances.append({
        'patient_id': patient_id,
        'label': label,
        'avg_chephy_distance': avg_distance,
        'num_sequences': len(sequences_set)
    })

results_df = pd.DataFrame(patient_distances)
print(results_df)
print(f"\nOverall average chephy distance: {results_df['avg_chephy_distance'].mean():.4f}")

      patient_id    label  avg_chephy_distance  num_sequences
0            OVI     sick             0.272949           4878
1      29.clones  healthy             0.271240           4828
2      47.clones  healthy             0.263672           4666
3      24.clones  healthy             0.268066           4543
4       7.clones  healthy             0.273682           4796
..           ...      ...                  ...            ...
77      5.clones  healthy             0.271729           4792
78   OVA3.clones     sick             0.270996           4847
79  OVA17.clones     sick             0.272949           4882
80           OVI     sick             0.267090           4873
81           OVI     sick             0.269043           4924

[82 rows x 4 columns]

Overall average chephy distance: 0.2705
